## 1. Loading the Dataset

In [50]:
import ir_datasets
import json
from collections import Counter
from collections import defaultdict
import statistics
import os

In [51]:
dataset = ir_datasets.load("cranfield")

docs = list(dataset.docs_iter())
print("="*100)
print(f"Total docs: {len(docs)}")
print("="*100)
print("First doc:")
print(f"  id: {docs[0].doc_id}")
print(f"  text: {docs[0].text[:300]}")
print("="*100)

Total docs: 1400
First doc:
  id: 1
  text: experimental investigation of the aerodynamics of a
wing in a slipstream .
  an experimental study of a wing in a propeller slipstream was
made in order to determine the spanwise distribution of the lift
increase due to slipstream at different angles of attack of the wing
and at different free strea


In [52]:
queries = list(dataset.queries_iter())
print("="*100)
print(f"Total queries: {len(queries)}")
print("="*100)
print("First query:")
print(f"  id: {queries[0].query_id}")
print(f"  text: {queries[0].text}")
print("="*100)

Total queries: 225
First query:
  id: 1
  text: what similarity laws must be obeyed when constructing aeroelastic models
of heated high speed aircraft .


In [53]:
qrels = list(dataset.qrels_iter())
print("="*100)
print(f"Total qrels: {len(qrels)}")
print("="*100)
print("First 5 qrels:")
for qrel in qrels[:5]:
    print(f"  query={qrel.query_id}, doc={qrel.doc_id}, relevance={qrel.relevance}")
print("="*100)

Total qrels: 1837
First 5 qrels:
  query=1, doc=184, relevance=2
  query=1, doc=29, relevance=2
  query=1, doc=31, relevance=2
  query=1, doc=12, relevance=3
  query=1, doc=51, relevance=3


## 2. Exploratory Analysis

In [54]:
rel_counts = Counter(qrel.relevance for qrel in qrels)
print(f"Relevance distribution: {dict(sorted(rel_counts.items()))}")

Relevance distribution: {-1: 225, 1: 128, 2: 387, 3: 734, 4: 363}


In [55]:
query_rels = defaultdict(list)
for qrel in qrels:
    query_rels[qrel.query_id].append(qrel.relevance)

rel_per_query = [
    sum(1 for r in rels if r >= 2) 
    for rels in query_rels.values()
] 
print(f"Min relevant per query: {min(rel_per_query)}")
print(f"Max relevant per query: {max(rel_per_query)}")
print(f"Median: {statistics.median(rel_per_query)}")
print(f"Queries with 0 relevant docs: {sum(1 for x in rel_per_query if x == 0)}")

Min relevant per query: 0
Max relevant per query: 39
Median: 5
Queries with 0 relevant docs: 10


In [56]:
bad_queries = [
    qid for qid, rels in query_rels.items()
    if sum(1 for r in rels if r >= 2) == 0
]
print("="*100)
print(f"Query id's with 0 relevant docs: {bad_queries}")
print("="*100)

query_dict = {q.query_id: q.text for q in queries}
for qid in bad_queries:
    print(f"Query {qid}: {query_dict[qid]}")
    print("="*100)

for qid in bad_queries:
    rels = query_rels[qid]
    print(f"Query {qid} relevance scores: {sorted(set(rels))}")

print("="*100)

Query id's with 0 relevant docs: ['22', '138', '142', '143', '165', '168', '169', '173', '192', '216']
Query 22: did anyone else discover that the turbulent skin friction is not over
sensitive to the nature of the variation of the viscosity with
temperature .
Query 138: has the effect of initial stresses,  on the frequencies of vibration of
circular cylindrical shells,  been investigated .
Query 142: what dome contours minimize discontinuity stresses when used as closures
on cylindrical pressure vessels .
Query 143: what general solutions for the stresses in pressurized shells of
revolution are available .
Query 165: are the stable profiles of a compressible boundary layer induced by a
moving wave known .
Query 168: what approximate solutions are known to the direct problem of transonic
flow in the throat of a nozzle,  i.e. finding the flow in a given
nozzle .
Query 169: what approximate solutions are known to the indirect problem of
transonic flow in the throat of a nozzle,  i.e. find

## 3. A Bit of Cleaning

In [57]:
BAD_QUERIES = {'22','138','142','143','165','168','169','173','192','216'}

docs_dict = {doc.doc_id: doc.text for doc in docs}
query_dict = {q.query_id: q.text for q in queries}


qrels_binary = defaultdict(dict)
qrels_graded = defaultdict(dict)

for qrel in qrels:
    if qrel.query_id in BAD_QUERIES:
        continue
    qrels_binary[qrel.query_id][qrel.doc_id] = int(qrel.relevance >= 2)
    qrels_graded[qrel.query_id][qrel.doc_id] = max(0, qrel.relevance)

print(f"Clean queries: {len(qrels_binary)}")
print(f"Total docs: {len(docs_dict)}")

min_rel = min(sum(v for v in rels.values()) for rels in qrels_binary.values())
print(f"Min relevant per query (after clean): {min_rel}")

Clean queries: 215
Total docs: 1400
Min relevant per query (after clean): 1


## 4. Saving

In [60]:
with open("docs.json", "w") as f:
    json.dump(docs_dict, f, indent=2)

with open("queries.json", "w") as f:
    json.dump(query_dict, f, indent=2)

with open("qrels_binary.json", "w") as f:
    json.dump(qrels_binary, f, indent=2)

with open("qrels_graded.json", "w") as f:
    json.dump(qrels_graded, f, indent=2)

print(f"docs.json - {len(docs_dict)} documents")
print(f"queries.json - {len(query_dict)} queries (all 225)")
print(f"qrels_binary.json - {len(qrels_binary)} queries (215 clean, for BIM/RSJ training)")
print(f"qrels_graded.json - {len(qrels_graded)} queries (215 clean, for NDCG evaluation)")

docs.json - 1400 documents
queries.json - 225 queries (all 225)
qrels_binary.json - 215 queries (215 clean, for BIM/RSJ training)
qrels_graded.json - 215 queries (215 clean, for NDCG evaluation)
